# Base Rate Calculation Engine Demonstration

This notebook demonstrates the Base Rate Calculation Engine, which aggregates Federal Reserve rates, Algorand DeFi protocol rates, and Algorand staking rewards to calculate risk-free base rates for lending.

## Features Demonstrated:
- Federal Reserve rate integration (Fed Funds, Treasury rates, SOFR)
- Algorand DeFi protocol rate aggregation (Folks Finance, Tinyman)
- Risk-free rate calculation with Algorand staking rewards
- Term structure modeling for different loan durations
- Real-time rate calculations with confidence scoring
- Historical data analysis and trending
- MCP integration for live data (when available)

In [ ]:
# Import required libraries
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Add the core module to path
sys.path.append('../')

# Import the base rate calculation engine
from core.config import BaseRateCalculationConfig, load_config
from core.base_rate_calculation import (
    BaseRateCalculationEngine,
    RateDataPoint,
    BaseRateResult,
    calculate_current_base_rate,
    get_base_rate_engine
)

# Set up plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("Base Rate Calculation Engine Demo")
print("=" * 50)
print(f"Notebook started at: {datetime.now()}")

## 1. Configuration and Setup

Let's start by loading the configuration and setting up the engine.

In [ ]:
# Load configuration
config = load_config()

# Display key configuration parameters
print("Configuration Summary:")
print("-" * 30)
print(f"MCP Algorand Reader URL: {config.mcp_services.algorand_reader_url}")
print(f"MCP Market Data URL: {config.mcp_services.market_data_url}")
print(f"Update Frequency: {config.calculation_parameters.update_frequency_minutes} minutes")
print(f"Database Path: {config.database.default_path}")
print(f"Minimum Rate: {config.calculation_parameters.bounds.minimum_rate:.3%}")
print(f"Maximum Rate: {config.calculation_parameters.bounds.maximum_rate:.3%}")

print("\nFederal Reserve Rate Sources:")
print(f"- Fed Funds Rate: {config.federal_reserve_rates.fed_funds_rate.enabled} (weight: {config.federal_reserve_rates.fed_funds_rate.weight})")
print(f"- Treasury 10Y: {config.federal_reserve_rates.treasury_10y.enabled} (weight: {config.federal_reserve_rates.treasury_10y.weight})")
print(f"- Treasury 3M: {config.federal_reserve_rates.treasury_3m.enabled} (weight: {config.federal_reserve_rates.treasury_3m.weight})")
print(f"- SOFR: {config.federal_reserve_rates.sofr_rate.enabled} (weight: {config.federal_reserve_rates.sofr_rate.weight})")

print("\nAlgorand DeFi Sources:")
print(f"- Folks Finance: {config.algorand_defi_rates.folks_finance.enabled} (weight: {config.algorand_defi_rates.folks_finance.weight})")
print(f"- Tinyman: {config.algorand_defi_rates.tinyman.enabled} (weight: {config.algorand_defi_rates.tinyman.weight})")
print(f"- Algorand Staking: {config.algorand_defi_rates.algorand_staking.enabled} (weight: {config.algorand_defi_rates.algorand_staking.weight})")

In [ ]:
# Initialize the base rate calculation engine
engine = BaseRateCalculationEngine(config)

# Perform health check
health = engine.health_check()
print("Engine Health Check:")
print("-" * 20)
print(f"Overall Status: {health['status']}")
for check_name, check_result in health['checks'].items():
    print(f"- {check_name}: {check_result}")
    
print(f"\nHealth check performed at: {health['timestamp']}")

## 2. Current Base Rate Calculation

Let's calculate the current base rate using all available data sources.

In [ ]:
# Calculate current base rate
print("Calculating current base rate...")
result = engine.calculate_base_rate()

print("\nBase Rate Calculation Result:")
print("=" * 40)
print(f"Base Rate: {result.base_rate:.4%}")
print(f"Risk-Free Rate: {result.risk_free_rate:.4%}")
print(f"Confidence Score: {result.confidence_score:.2%}")
print(f"Calculation Time: {result.calculation_timestamp}")
print(f"Data Sources: {', '.join(result.data_sources)}")

if 'calculation_time_ms' in result.metadata:
    print(f"Calculation Duration: {result.metadata['calculation_time_ms']:.1f} ms")
if 'num_data_points' in result.metadata:
    print(f"Data Points Used: {result.metadata['num_data_points']}")

In [ ]:
# Display component rates breakdown
print("\nComponent Rates Breakdown:")
print("-" * 30)
for component, rate in result.component_rates.items():
    if isinstance(rate, (int, float)):
        print(f"{component}: {rate:.4%}")
    else:
        print(f"{component}: {rate}")

# Create a DataFrame for better visualization
component_df = pd.DataFrame([
    {'Component': k, 'Rate': v} 
    for k, v in result.component_rates.items() 
    if isinstance(v, (int, float))
])

if not component_df.empty:
    # Plot component rates
    plt.figure(figsize=(12, 6))
    bars = plt.bar(component_df['Component'], component_df['Rate'] * 100)
    plt.title('Base Rate Components', fontsize=16, fontweight='bold')
    plt.xlabel('Rate Component')
    plt.ylabel('Rate (%)')
    plt.xticks(rotation=45, ha='right')
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{height:.3f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
else:
    print("No numeric component rates to display")

## 3. Term Structure Analysis

Let's examine the term structure (yield curve) for different loan durations.

In [ ]:
# Display term structure
print("Term Structure (Yield Curve):")
print("-" * 30)
for term, rate in result.term_structure.items():
    print(f"{term.replace('_', ' ').title()}: {rate:.4%}")

# Create term structure visualization
if result.term_structure:
    # Convert term structure to DataFrame
    term_df = pd.DataFrame([
        {'Term': k.replace('_', ' ').title(), 'Rate': v, 'Sort_Key': i} 
        for i, (k, v) in enumerate(result.term_structure.items())
    ])
    
    # Sort by original order
    term_df = term_df.sort_values('Sort_Key')
    
    # Plot yield curve
    plt.figure(figsize=(14, 8))
    
    # Main yield curve
    plt.subplot(2, 1, 1)
    plt.plot(range(len(term_df)), term_df['Rate'] * 100, 'o-', linewidth=2, markersize=8)
    plt.title('Yield Curve (Term Structure)', fontsize=16, fontweight='bold')
    plt.xlabel('Term')
    plt.ylabel('Interest Rate (%)')
    plt.xticks(range(len(term_df)), term_df['Term'], rotation=45, ha='right')
    plt.grid(True, alpha=0.3)
    
    # Add rate labels
    for i, rate in enumerate(term_df['Rate']):
        plt.text(i, rate * 100 + 0.01, f'{rate:.3%}', ha='center', va='bottom')
    
    # Term premiums (spread over base rate)
    plt.subplot(2, 1, 2)
    term_premiums = [(rate - result.base_rate) * 100 for rate in term_df['Rate']]
    bars = plt.bar(range(len(term_df)), term_premiums)
    plt.title('Term Premiums (Spread over Base Rate)', fontsize=14, fontweight='bold')
    plt.xlabel('Term')
    plt.ylabel('Premium (%)')
    plt.xticks(range(len(term_df)), term_df['Term'], rotation=45, ha='right')
    plt.grid(True, alpha=0.3)
    
    # Add premium labels
    for bar, premium in zip(bars, term_premiums):
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.001,
                f'{premium:.3f}%', ha='center', va='bottom' if height >= 0 else 'top')
    
    plt.tight_layout()
    plt.show()

## 4. Historical Rate Analysis

Let's perform multiple calculations to build historical data and analyze trends.

In [ ]:
# Perform multiple calculations to simulate historical data
print("Generating historical rate calculations...")
historical_results = []

for i in range(10):
    result = engine.calculate_base_rate()
    historical_results.append({
        'timestamp': result.calculation_timestamp,
        'base_rate': result.base_rate,
        'risk_free_rate': result.risk_free_rate,
        'confidence_score': result.confidence_score,
        'calculation_time_ms': result.metadata.get('calculation_time_ms', 0)
    })
    
    # Small delay to create timestamp differences
    import time
    time.sleep(0.1)

# Convert to DataFrame
hist_df = pd.DataFrame(historical_results)
hist_df['timestamp'] = pd.to_datetime(hist_df['timestamp'])

print(f"Generated {len(hist_df)} historical calculations")
print("\nSummary Statistics:")
print(hist_df[['base_rate', 'risk_free_rate', 'confidence_score']].describe())

In [ ]:
# Visualize historical trends
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Base rate over time
axes[0, 0].plot(hist_df['timestamp'], hist_df['base_rate'] * 100, 'o-', linewidth=2)
axes[0, 0].set_title('Base Rate Over Time')
axes[0, 0].set_ylabel('Rate (%)')
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

# Risk-free rate over time
axes[0, 1].plot(hist_df['timestamp'], hist_df['risk_free_rate'] * 100, 'o-', color='orange', linewidth=2)
axes[0, 1].set_title('Risk-Free Rate Over Time')
axes[0, 1].set_ylabel('Rate (%)')
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].tick_params(axis='x', rotation=45)

# Confidence score over time
axes[1, 0].plot(hist_df['timestamp'], hist_df['confidence_score'] * 100, 'o-', color='green', linewidth=2)
axes[1, 0].set_title('Confidence Score Over Time')
axes[1, 0].set_ylabel('Confidence (%)')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].tick_params(axis='x', rotation=45)

# Calculation time distribution
axes[1, 1].hist(hist_df['calculation_time_ms'], bins=10, alpha=0.7, color='purple')
axes[1, 1].set_title('Calculation Time Distribution')
axes[1, 1].set_xlabel('Time (ms)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Rate Source Analysis

Let's analyze the individual rate sources and their contributions.

In [ ]:
# Test individual rate sources
print("Testing Individual Rate Sources:")
print("=" * 40)

# Federal Reserve rates
print("\n1. Federal Reserve Rates:")
fed_rates = engine.fetch_federal_reserve_rates()
for rate in fed_rates:
    print(f"   {rate.rate_type}: {rate.value:.4%} (confidence: {rate.confidence:.1%}, source: {rate.source})")

# Algorand ecosystem rates
print("\n2. Algorand Ecosystem Rates:")
algo_rates = engine.fetch_algorand_rates()
for rate in algo_rates:
    print(f"   {rate.rate_type}: {rate.value:.4%} (confidence: {rate.confidence:.1%}, source: {rate.source})")

# Create source comparison visualization
all_rates = fed_rates + algo_rates
if all_rates:
    rate_df = pd.DataFrame([
        {
            'Source': rate.source,
            'Rate_Type': rate.rate_type,
            'Rate': rate.value,
            'Confidence': rate.confidence
        }
        for rate in all_rates
    ])
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    # Rates by source
    source_rates = rate_df.groupby('Source')['Rate'].mean() * 100
    bars1 = axes[0].bar(range(len(source_rates)), source_rates.values)
    axes[0].set_title('Average Rates by Source')
    axes[0].set_ylabel('Rate (%)')
    axes[0].set_xticks(range(len(source_rates)))
    axes[0].set_xticklabels(source_rates.index, rotation=45, ha='right')
    axes[0].grid(True, alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars1, source_rates.values):
        axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                    f'{value:.3f}%', ha='center', va='bottom')
    
    # Confidence by source
    source_conf = rate_df.groupby('Source')['Confidence'].mean() * 100
    bars2 = axes[1].bar(range(len(source_conf)), source_conf.values, color='orange')
    axes[1].set_title('Average Confidence by Source')
    axes[1].set_ylabel('Confidence (%)')
    axes[1].set_xticks(range(len(source_conf)))
    axes[1].set_xticklabels(source_conf.index, rotation=45, ha='right')
    axes[1].grid(True, alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars2, source_conf.values):
        axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                    f'{value:.1f}%', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()

## 6. Configuration Scenario Testing

Let's test different market scenarios and configuration parameters.

In [ ]:
# Test different market scenarios from config
scenarios = config.development.test_scenarios

print("Testing Market Scenarios:")
print("=" * 30)

scenario_results = []

for scenario_name, scenario_params in scenarios.items():
    print(f"\n{scenario_name.replace('_', ' ').title()}:")
    
    # Display scenario parameters
    for param, value in scenario_params.items():
        if isinstance(value, (int, float)) and param != 'expected_base_rate_range':
            print(f"  {param}: {value:.3%}")
        elif param == 'expected_base_rate_range':
            print(f"  Expected range: {value[0]:.3%} - {value[1]:.3%}")
    
    # For demonstration, we'll use our current calculation
    # In practice, you might modify the engine with scenario-specific rates
    result = engine.calculate_base_rate()
    
    scenario_results.append({
        'scenario': scenario_name,
        'base_rate': result.base_rate,
        'confidence': result.confidence_score
    })
    
    print(f"  Calculated base rate: {result.base_rate:.3%}")
    print(f"  Confidence: {result.confidence_score:.1%}")

# Visualize scenario results
scenario_df = pd.DataFrame(scenario_results)

plt.figure(figsize=(12, 6))
x_pos = range(len(scenario_df))
bars = plt.bar(x_pos, scenario_df['base_rate'] * 100)
plt.title('Base Rates Across Market Scenarios', fontsize=16, fontweight='bold')
plt.xlabel('Market Scenario')
plt.ylabel('Base Rate (%)')
plt.xticks(x_pos, [s.replace('_', ' ').title() for s in scenario_df['scenario']], rotation=45, ha='right')

# Add confidence as text labels
for i, (bar, conf) in enumerate(zip(bars, scenario_df['confidence'])):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f'{height:.3f}%\n(conf: {conf:.1%})', ha='center', va='bottom', fontsize=9)

plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Performance and Efficiency Analysis

In [ ]:
# Performance testing
import time

print("Performance Analysis:")
print("=" * 20)

# Test calculation speed
num_calculations = 20
calculation_times = []

print(f"Performing {num_calculations} rate calculations...")

for i in range(num_calculations):
    start_time = time.time()
    result = engine.calculate_base_rate()
    end_time = time.time()
    
    calculation_time = (end_time - start_time) * 1000  # Convert to milliseconds
    calculation_times.append(calculation_time)

# Performance statistics
perf_stats = {
    'mean_time_ms': np.mean(calculation_times),
    'median_time_ms': np.median(calculation_times),
    'min_time_ms': np.min(calculation_times),
    'max_time_ms': np.max(calculation_times),
    'std_time_ms': np.std(calculation_times)
}

print("\nPerformance Statistics:")
for stat, value in perf_stats.items():
    print(f"{stat.replace('_', ' ').title()}: {value:.2f} ms")

# Performance visualization
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Calculation times over iterations
axes[0].plot(range(1, num_calculations + 1), calculation_times, 'o-', linewidth=2)
axes[0].axhline(y=perf_stats['mean_time_ms'], color='red', linestyle='--', alpha=0.7, label=f"Mean: {perf_stats['mean_time_ms']:.1f} ms")
axes[0].set_title('Calculation Time per Iteration')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Time (ms)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Calculation time distribution
axes[1].hist(calculation_times, bins=10, alpha=0.7, color='green')
axes[1].axvline(x=perf_stats['mean_time_ms'], color='red', linestyle='--', alpha=0.7, label=f"Mean: {perf_stats['mean_time_ms']:.1f} ms")
axes[1].set_title('Calculation Time Distribution')
axes[1].set_xlabel('Time (ms)')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Database and Historical Analysis

In [ ]:
# Get historical data from database
historical_df = engine.get_historical_rates(days=1)

print(f"Historical Data Analysis:")
print(f"Records in database: {len(historical_df)}")

if len(historical_df) > 0:
    print("\nData Summary:")
    print(historical_df[['base_rate', 'risk_free_rate', 'confidence_score']].describe())
    
    # Plot historical trends if we have data
    if len(historical_df) > 1:
        historical_df['timestamp'] = pd.to_datetime(historical_df['timestamp'])
        historical_df = historical_df.sort_values('timestamp')
        
        plt.figure(figsize=(15, 8))
        
        # Plot rates over time
        plt.subplot(2, 1, 1)
        plt.plot(historical_df['timestamp'], historical_df['base_rate'] * 100, 'o-', label='Base Rate', linewidth=2)
        plt.plot(historical_df['timestamp'], historical_df['risk_free_rate'] * 100, 'o-', label='Risk-Free Rate', linewidth=2)
        plt.title('Historical Rate Trends', fontsize=16, fontweight='bold')
        plt.ylabel('Rate (%)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        
        # Plot confidence scores
        plt.subplot(2, 1, 2)
        plt.plot(historical_df['timestamp'], historical_df['confidence_score'] * 100, 'o-', color='green', linewidth=2)
        plt.title('Confidence Score Over Time', fontsize=14, fontweight='bold')
        plt.xlabel('Time')
        plt.ylabel('Confidence (%)')
        plt.grid(True, alpha=0.3)
        plt.xticks(rotation=45)
        
        plt.tight_layout()
        plt.show()
    
    # Show latest records
    print("\nLatest 5 Records:")
    latest_records = historical_df.head(5)[['timestamp', 'base_rate', 'risk_free_rate', 'confidence_score']]
    latest_records['base_rate'] = latest_records['base_rate'].apply(lambda x: f"{x:.4%}")
    latest_records['risk_free_rate'] = latest_records['risk_free_rate'].apply(lambda x: f"{x:.4%}")
    latest_records['confidence_score'] = latest_records['confidence_score'].apply(lambda x: f"{x:.2%}")
    print(latest_records.to_string(index=False))
else:
    print("No historical data available in database.")

## 9. MCP Integration Testing

Let's test the MCP service integration if services are available.

In [ ]:
# Test MCP service connectivity
import requests

print("MCP Service Integration Testing:")
print("=" * 35)

mcp_status = {}

# Test Algorand Reader MCP (port 8002)
try:
    response = requests.get(f"{config.mcp_services.algorand_reader_url}/health", timeout=3)
    mcp_status['algorand_reader'] = {
        'status': 'available' if response.status_code == 200 else 'error',
        'status_code': response.status_code
    }
except requests.RequestException as e:
    mcp_status['algorand_reader'] = {'status': 'unavailable', 'error': str(e)}

# Test Market Data MCP (port 8003)
try:
    response = requests.get(f"{config.mcp_services.market_data_url}/health", timeout=3)
    mcp_status['market_data'] = {
        'status': 'available' if response.status_code == 200 else 'error',
        'status_code': response.status_code
    }
except requests.RequestException as e:
    mcp_status['market_data'] = {'status': 'unavailable', 'error': str(e)}

# Display MCP status
for service, status in mcp_status.items():
    print(f"\n{service.replace('_', ' ').title()} Service:")
    print(f"  Status: {status['status']}")
    if 'status_code' in status:
        print(f"  Status Code: {status['status_code']}")
    if 'error' in status:
        print(f"  Error: {status['error']}")

# Test data retrieval if services are available
available_services = [k for k, v in mcp_status.items() if v['status'] == 'available']

if available_services:
    print(f"\nTesting data retrieval from {len(available_services)} available service(s)...")
    
    # Example: Test getting ALGO price if market data service is available
    if 'market_data' in available_services:
        try:
            response = requests.get(f"{config.mcp_services.market_data_url}/price/ALGO", timeout=5)
            if response.status_code == 200:
                price_data = response.json()
                print(f"  ALGO Price: ${price_data.get('price', 'N/A')}")
            else:
                print(f"  Market data request failed: {response.status_code}")
        except Exception as e:
            print(f"  Market data request error: {e}")
else:
    print("\nNo MCP services available for testing.")
    print("The engine will use simulated/fallback data for rate calculations.")

# Visualization of service status
service_names = list(mcp_status.keys())
service_statuses = [1 if status['status'] == 'available' else 0 for status in mcp_status.values()]

plt.figure(figsize=(10, 5))
colors = ['green' if status == 1 else 'red' for status in service_statuses]
bars = plt.bar(service_names, service_statuses, color=colors, alpha=0.7)
plt.title('MCP Service Availability Status', fontsize=16, fontweight='bold')
plt.ylabel('Available (1) / Unavailable (0)')
plt.ylim(0, 1.2)

# Add status labels
for bar, status_val, service in zip(bars, service_statuses, service_names):
    status_text = 'Available' if status_val == 1 else 'Unavailable'
    plt.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.05,
            status_text, ha='center', va='bottom', fontweight='bold')

plt.xticks([name.replace('_', ' ').title() for name in service_names])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Summary and Recommendations

Let's summarize the engine performance and provide recommendations.

In [ ]:
# Generate comprehensive summary
print("Base Rate Calculation Engine Summary")
print("=" * 50)

# Get final calculation
final_result = engine.calculate_base_rate()

print(f"\nCurrent Base Rate: {final_result.base_rate:.4%}")
print(f"Risk-Free Rate: {final_result.risk_free_rate:.4%}")
print(f"Confidence Score: {final_result.confidence_score:.2%}")
print(f"Data Sources: {len(final_result.data_sources)}")

# Engine health summary
health = engine.health_check()
print(f"\nEngine Health: {health['status']}")

# Performance summary
if 'calculation_time_ms' in final_result.metadata:
    print(f"Calculation Performance: {final_result.metadata['calculation_time_ms']:.1f} ms")

# MCP integration summary
available_mcps = sum(1 for status in mcp_status.values() if status['status'] == 'available')
total_mcps = len(mcp_status)
print(f"MCP Integration: {available_mcps}/{total_mcps} services available")

print("\n" + "=" * 50)
print("RECOMMENDATIONS:")
print("=" * 50)

recommendations = []

# Performance recommendations
if perf_stats['mean_time_ms'] > 1000:
    recommendations.append("⚠️  Consider optimizing calculation performance (>1s average)")
else:
    recommendations.append("✅ Calculation performance is acceptable")

# Confidence recommendations
if final_result.confidence_score < 0.8:
    recommendations.append("⚠️  Consider improving data sources for higher confidence")
else:
    recommendations.append("✅ Rate calculation confidence is high")

# MCP integration recommendations
if available_mcps == 0:
    recommendations.append("⚠️  No MCP services available - start MCP services for real-time data")
elif available_mcps < total_mcps:
    recommendations.append(f"⚠️  Only {available_mcps}/{total_mcps} MCP services available")
else:
    recommendations.append("✅ All MCP services are operational")

# Rate level recommendations
if final_result.base_rate < 0.01:
    recommendations.append("📊 Base rate is very low - consider market conditions")
elif final_result.base_rate > 0.15:
    recommendations.append("📊 Base rate is high - validate against market conditions")
else:
    recommendations.append("📊 Base rate appears reasonable for current market")

# Term structure recommendations
term_spread = max(final_result.term_structure.values()) - min(final_result.term_structure.values())
if term_spread < 0.01:
    recommendations.append("📈 Yield curve is flat - consider term premium adjustments")
elif term_spread > 0.05:
    recommendations.append("📈 Yield curve is steep - validate term structure parameters")
else:
    recommendations.append("📈 Term structure appears well-calibrated")

# Display recommendations
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

print("\n" + "=" * 50)
print("NEXT STEPS:")
print("=" * 50)
print("1. 🚀 Deploy MCP services for real-time data integration")
print("2. 🔧 Configure Federal Reserve API key for live FRED data")
print("3. 📊 Integrate with market conditions monitor for dynamic adjustments")
print("4. 🔄 Set up automated rate calculation scheduling")
print("5. 📈 Implement rate change alerts and monitoring")
print("6. 🧪 Validate rates against production lending scenarios")

print(f"\nDemo completed at: {datetime.now()}")

## Conclusion

This demonstration showcased the Base Rate Calculation Engine's capabilities:

### ✅ Key Features Demonstrated:
- **Multi-source rate aggregation**: Federal Reserve + Algorand ecosystem
- **Risk-free rate calculation**: Weighted composite with adjustments
- **Term structure modeling**: Yield curve for different loan durations
- **Real-time calculations**: Fast, reliable rate computation
- **Confidence scoring**: Data quality and reliability assessment
- **Historical tracking**: Database storage and trend analysis
- **MCP integration**: Ready for live market data
- **Performance monitoring**: Health checks and optimization

### 🎯 Production Readiness:
- Configurable via YAML files
- Environment variable overrides
- Comprehensive error handling
- Database persistence
- Health monitoring
- Performance tracking

### 🔗 Integration Points:
- Market Conditions Monitor (for dynamic rate adjustments)
- Lending Platform (for rate application)
- Risk Management Systems
- Regulatory Reporting

The engine is ready for integration into the larger Algorand lending ecosystem!